# Lezione 9 — Google Colab, GPU/TPU e migrazione da TensorFlow 1.x

Corso **"Fondamenti di TensorFlow da zero"** — DIH DATAlife

## Obiettivi di questa lezione

- Usare Google Colab per addestrare modelli senza installare nulla in locale
- Sfruttare GPU e TPU gratuiti (o a basso costo) in Colab
- Migrare codice da TensorFlow 1.x a TensorFlow 2.x
- Comprendere la strategia di distribuzione (distributed training) per addestrare su più GPU/TPU

## 1. Che cos'è Google Colaboratory

Colab è un notebook Jupyter che gira su un server remoto di Google. Accedi da qualsiasi browser, scrivi Python e aggiungi celle di testo (Markdown) come in un notebook Jupyter locale. Google fornisce gratuitamente una GPU K80 o TPU per ogni sessione (con limiti di tempo).

In [ ]:
import tensorflow as tf

print("Versión de TensorFlow:", tf.__version__)
print("Ejecución eager activada:", tf.executing_eagerly())

### Montare Google Drive (opzionale, richiede autorizzazioni del tuo account)

Questa cella ti chiede il permesso di leggere e scrivere file nel tuo Google Drive. Una volta autorizzata, puoi leggere file dal Drive direttamente nel codice.

In [ ]:
# Descomenta estas líneas si estás en Colab y quieres montar tu Drive
# from google.colab import drive
# drive.mount('/content/drive')
# print("Drive montado. Ejemplo de ruta: /content/drive/MyDrive/mi_carpeta/mis_datos.csv")

### Caricare un file scelto (opzionale, richiede Colab)

È utile quando hai bisogno di un solo file dal tuo computer, senza montare tutto il Drive.

In [ ]:
# Descomenta estas líneas si estás en Colab y quieres subir un archivo
# from google.colab import files
# subidos = files.upload()
# print("Archivos subidos:", list(subidos.keys()))

## 2. Controllare quale dispositivo hai disponibile: CPU, GPU o TPU

`tf.config.list_physical_devices()` mostra che cosa Colab ha allocato per la tua sessione.

In [ ]:
print("CPU (Central Processing Unit):", tf.config.list_physical_devices('CPU'))
print("GPU (Graphics Processing Unit):", tf.config.list_physical_devices('GPU'))
print("TPU (Tensor Processing Unit):", tf.config.list_physical_devices('TPU'))

**Per vedere una GPU nell'elenco precedente**: nel menu di Colab vai a **Ambiente di esecuzione** > **Cambia tipo di runtime** e seleziona **GPU**.

(Se scegli "TPU", otterrai un'unità di elaborazione del tensore di Google, ancora più veloce ma con API diverse.)

## 3. Usare la GPU nell'addestramento

Con GPU attivata, TensorFlow la usa automaticamente. Qualsiasi operazione su GPU è generalmente 10-100 volte più veloce che su CPU.

In [ ]:
import time

# Datos pequeños de ejemplo: no hace falta un dataset real para ver el patrón
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0

def crear_modelo():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

dispositivo = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print("Entrenando en:", dispositivo)

with tf.device(dispositivo):
    modelo = crear_modelo()
    modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    inicio = time.time()
    modelo.fit(x_train, y_train, epochs=1, batch_size=128, verbose=0)
    print(f"Tiempo de entrenamiento: {time.time() - inicio:.2f} segundos")

## 4. Usare una TPU: idea semplificata dell'esempio di Shakespeare

Il notebook originale della certificazione EITC/AI/TFF (`Shakespeare_with_Cloud_TPUs.ipynb`) addestra un modello su milioni di righe di Shakespeare usando una TPU v2 (con 8 core).

In [ ]:
import os

# --- CELDA QUE REQUIERE TPU REAL EN COLAB ---
if 'COLAB_TPU_ADDR' in os.environ:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(
        tpu='grpc://' + os.environ['COLAB_TPU_ADDR']
    )
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.experimental.TPUStrategy(resolver)
    print("TPU conectada. Dispositivos lógicos:", tf.config.list_logical_devices('TPU'))
else:
    print("No hay TPU activada en este entorno: activa TPU en el menú Entorno de ejecución")
    strategy = tf.distribute.get_strategy()  # estrategia "normal", sin distribución

La cella seguente sì viene eseguita sempre, con o senza TPU: se rileviamo TPU usiamo la distribuzione con TPU, altrimenti usiamo solo la CPU.

In [ ]:
# Modelo LSTM muy reducido, solo para ilustrar el patrón de distribución
# (en el notebook original, EMBEDDING_DIM=512 y se entrena sobre el texto
# completo de Shakespeare durante 10 épocas; aquí lo simplificamos a un
# vocabulario y una secuencia diminutos para que corra en segundos en CPU)

VOCAB_SIZE = 256   # un carácter ASCII por token, como en el notebook original
EMBEDDING_DIM = 32 # reducido respecto al original (512) para ir rápido sin TPU
SEQ_LEN = 20

def lstm_model():
    entrada = tf.keras.Input(shape=(SEQ_LEN,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(entrada)
    x = tf.keras.layers.LSTM(EMBEDDING_DIM, return_sequences=True)(x)
    salida = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax')
    )(x)
    return tf.keras.Model(inputs=entrada, outputs=salida)

with strategy.scope():
    modelo_texto = lstm_model()
    modelo_texto.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')

# Datos aleatorios solo para comprobar que el ciclo de entrenamiento funciona
import numpy as np
x_ejemplo = np.random.randint(0, VOCAB_SIZE, size=(64, SEQ_LEN))
y_ejemplo = np.random.randint(0, VOCAB_SIZE, size=(64, SEQ_LEN, 1))

modelo_texto.fit(x_ejemplo, y_ejemplo, epochs=1, batch_size=16, verbose=1)
print("Entrenamiento de ejemplo completado (con o sin TPU, el código es el mismo)")

## 5. Differenze di sintassi: TensorFlow 1.x contro 2.x

Il cambio più importante: TensorFlow 1.x costruisce un grafo computazionale in una prima fase, e lo esegue in una sessione. TensorFlow 2.x esegue il codice direttamente (esecuzione eager) come il Python normale.

In [ ]:
# Estilo TensorFlow 1.x, usando el módulo de compatibilidad tensorflow.compat.v1
# Construimos las operaciones dentro de un grafo explícito (en lugar de con
# tf1.disable_eager_execution(), que afectaría a todo el notebook a partir de
# aquí) para poder comparar ambos estilos sin romper las celdas siguientes.
import tensorflow.compat.v1 as tf1

grafo_v1 = tf.Graph()
with grafo_v1.as_default():
    a1 = tf1.constant(2)
    b1 = tf1.constant(3)
    c1 = tf1.add(a1, b1)

with tf1.Session(graph=grafo_v1) as sess:
    resultado = sess.run(c1)
    print("Resultado (estilo TF 1.x, con Session):", resultado)

In [ ]:
# Estilo TensorFlow 2.x: sin sesión, sin placeholders, sin inicializar variables a mano
a2 = tf.constant(2)
b2 = tf.constant(3)
c2 = tf.add(a2, b2)
print("Resultado (estilo TF 2.x, eager):", c2.numpy())

# Una variable está lista para usarse en el mismo momento en que la creas
peso = tf.Variable(0.5, name='peso')
print("Variable lista sin inicializador aparte:", peso.numpy())

Altre differenze che vale la pena ricordare:

- TensorFlow 1.x usava `tf.placeholder` e `tf.Variable` espliciti; TensorFlow 2.x ha gestione dello stato più semplice
- TensorFlow 1.x richiedeva `sess = tf.Session(); sess.run(...)` per ogni operazione; TensorFlow 2.x calcola ogni cosa al momento
- `tf.function` in TensorFlow 2.x converte una funzione Python in un grafo ottimizzato se hai bisogno di velocità extra, ma è opzionale
- Keras era inizialmente una libreria separata; TensorFlow 2.x lo incorpora come `tf.keras` (API ufficiale)

## Riepilogo

- Colab ti offre un ambiente di esecuzione remoto con TensorFlow preinstallato
- Puoi accedere a GPU e TPU gratuitamente (con limiti di tempo)
- Google Drive può essere montato per accedere ai tuoi file
- TensorFlow 2.x usa esecuzione eager per impostazione predefinita, semplificando il codice rispetto a 1.x
- `tf.function` e distribuzione sono opzionali per ottimizzare le prestazioni